In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar, minimize
import math

In [3]:
def f1(x):
    return np.log(1 + x**2) + x**3

def f2(x):
    return x ** 5 + 0.4 * np.sign(x) * x ** 4 + 2

In [4]:
def f1_der_5(x):
    return 48 * x * (1 - 5 * x ** 2 + x ** 4) / ((1 + x ** 2) ** 5)

def f2_der_5(x):
    return 120 * x ** 0

In [5]:
fig = go.Figure()
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("f1", "f2"),
)
fig.add_trace(go.Scatter(x=np.linspace(-2, 2, 1000), y=f1_der_5(np.linspace(-2, 2, 1000)), name='f1_der_5'), row=1, col=1)
fig.add_trace(go.Scatter(x=np.linspace(-2, 2, 1000), y=f2_der_5(np.linspace(-2, 2, 1000)), name='f2_der_5'), row=1, col=2)
fig.update_layout(title='5 производная функции', width=1000, height=600)
fig.show()


In [6]:
result_max = minimize_scalar(lambda x: -f1_der_5(x), method='bounded', bounds=(-0.5, 0.5))
result_max.fun = -result_max.fun
result_max

 message: Solution found.
 success: True
  status: 0
     fun: 6.3489378700059165
       x: 0.2121949817364503
     nit: 9
    nfev: 9

In [7]:
def load_data(filename):
    data = np.loadtxt(filename)
    return data[:,0], data[:,1], data[:,2], data[:,3]

def load_perturb(filename):
    data = np.loadtxt(filename)
    return data[:,0], data[:,1]

def theoretical_error(x_dense, x_nodes, M, n):
    w = np.ones_like(x_dense)
    for xi in x_nodes:
        w *= (x_dense - xi)

    return np.abs(w) * M / math.factorial(n)

In [216]:
def plot_interpolation(root, prefix, func, nodes_list):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Равномерная сетка (Uniform)", "Чебышевская сетка (Chebyshev)"),
    )

    for n in nodes_list[::-1]:
        x_u, f_u, p_u, err_u = load_data(f"{root}{prefix}_uniform_n{n}.txt")
        fig.add_trace(
            go.Scatter(x=x_u, y=p_u, mode='lines', name=f'P(x) n={n}', legendgroup="uniform",
                       line=dict(width=2)),
            row=1, col=1
        )

        x_c, f_c, p_c, err_c = load_data(f"{root}{prefix}_cheb_n{n}.txt")
        fig.add_trace(
            go.Scatter(x=x_c, y=p_c, mode='lines', name=f'P(x) n={n}', legendgroup="cheb",
                       line=dict(width=2)),
            row=1, col=2
        )

    fig.add_trace(
        go.Scatter(x=x_u, y=f_u, mode='lines', name='f(x)', legendgroup="uniform",
                   line=dict(color='black', width=3)),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=x_c, y=f_c, mode='lines', name='f(x)', legendgroup="cheb",
                   line=dict(color='black', width=3)),
        row=1, col=2
    )

    n = nodes_list[0]
    x_nodes_u = np.linspace(-2, 2, n)
    y_nodes_u = func(x_nodes_u)
    fig.add_trace(
        go.Scatter(x=x_nodes_u, y=y_nodes_u, mode='markers',
                   name=f'Узлы (n={n})', legendgroup="uniform",
                   marker=dict(color='red', size=9, line=dict(color='darkred', width=1))),
        row=1, col=1
    )

    x_nodes_c = -2 * np.cos(np.pi * (2 * np.arange(n) + 1) / (2 * n))
    y_nodes_c = func(x_nodes_c)
    fig.add_trace(
        go.Scatter(x=x_nodes_c, y=y_nodes_c, mode='markers',
                   name=f'Узлы (n={n})', legendgroup="cheb",
                   marker=dict(color='red', size=9, line=dict(color='darkred', width=1))),
        row=1, col=2
    )

    fig.update_layout(
        title=f"{prefix}: функция и интерполяционные полиномы",
        xaxis_title="x",
        yaxis_title="y",
        template="ggplot2",
        width=1000,
        height=600,
        legend_tracegroupgap=100,
        hovermode="x unified"
    )

    fig.show()


def plot_errors_with_theory(root, prefix, M, nodes_list):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Ошибка — Равномерная сетка", "Ошибка — Чебышевская сетка"),
    )

    for n in nodes_list:
        x_u, f_u, p_u, err_u = load_data(f"{root}{prefix}_uniform_n{n}.txt")
        fig.add_trace(
            go.Scatter(x=x_u, y=err_u, mode='lines', name=f'n={n}', legendgroup='uniform'),
            row=1, col=1
        )

        x_c, f_c, p_c, err_c = load_data(f"{root}{prefix}_cheb_n{n}.txt")
        fig.add_trace(
            go.Scatter(x=x_c, y=err_c, mode='lines', name=f'n={n}', legendgroup='cheb'),
            row=1, col=2
        )

    n_min = nodes_list[0]
    x_u, _, _, _ = load_data(f"{root}{prefix}_uniform_n{n_min}.txt")
    x_nodes_u = np.linspace(-2, 2, n_min)

    theory_u = theoretical_error(x_u, x_nodes_u, M, n_min)

    fig.add_trace(
        go.Scatter(x=x_u, y=theory_u, mode='lines', name='Теор. оценка',
                   line=dict(color='black', dash='dash', width=2), legendgroup='uniform'),
        row=1, col=1
    )

    x_nodes_c = -2 * np.cos(np.pi * (2 * np.arange(n_min) + 1) / (2 * n_min))
    theory_c = theoretical_error(x_u, x_nodes_c, M, n_min)

    fig.add_trace(
        go.Scatter(x=x_u, y=theory_c, mode='lines', name='Теор. оценка',
                   line=dict(color='black', dash='dash', width=2), legendgroup='cheb'),
        row=1, col=2
    )

    fig.update_yaxes(
        title_text="Ошибка |f(x) - P(x)|",
        row=1, col=1,
        matches='y'
    )
    fig.update_yaxes(
        title_text="Ошибка |f(x) - P(x)|",
        row=1, col=2,
        matches='y'
    )

    fig.update_layout(
        title=f"{prefix}: Сравнение ошибок интерполяции",
        height=650,
        width=1200,
        template="ggplot2",
        legend_tracegroupgap=100,
        hovermode="x unified",
    )

    fig.show()


def plot_error_vs_n(dir, prefix):

    data = np.loadtxt(dir + prefix + "_error.txt")

    n = data[:,0]
    err_uniform = data[:,1]
    err_cheb = data[:,2]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=n, y=err_uniform,
        mode='lines+markers',
        name='Равномерная сетка',
        marker=dict(size=8),
    ))
    fig.add_trace(go.Scatter(
        x=n, y=err_cheb,
        mode='lines+markers',
        name='Чебышевская сетка',
        marker=dict(size=8),
    ))

    idx_min_u = np.argmin(err_uniform)
    n_min_u = int(n[idx_min_u])
    err_min_u = err_uniform[idx_min_u]
    
    idx_min_c = np.argmin(err_cheb)
    n_min_c = int(n[idx_min_c])
    err_min_c = err_cheb[idx_min_c]
    

    fig.add_trace(go.Scatter(
        x=[n_min_u], y=[err_min_u],
        mode='markers',
        name='Минимум uniform',
        marker=dict(color='red', size=16, symbol='star', line=dict(color='darkred', width=2.5)),
        hovertemplate=f'МИНИМУМ (uniform)<br>n={n_min_u}<br>err={err_min_u:.2e}<extra></extra>'
    ))
    
    fig.add_trace(go.Scatter(
        x=[n_min_c], y=[err_min_c],
        mode='markers',
        name='Минимум Chebyshev',
        marker=dict(color='blue', size=16, symbol='star', line=dict(color='darkblue', width=2.5)),
        hovertemplate=f'МИНИМУМ (Chebyshev)<br>n={n_min_c}<br>err={err_min_c:.2e}<extra></extra>'
    ))
    
    fig.add_annotation(
        x=n_min_u,
        y=np.log10(err_min_u),
        text=f"МИНИМУМ<br>uniform<br>n = {n_min_u}<br>err = {err_min_u:.2e}",
        showarrow=True,
        arrowhead=2,
        arrowsize=1.5,
        arrowwidth=2,
        arrowcolor="red",
        ax=40, ay=-60,
        bgcolor="rgba(255,255,255,0.5)",
        bordercolor="red",
        borderwidth=2,
        borderpad=4,
        font=dict(size=13, color="darkred", family="Arial Black"),
        align="left"
    )
    
    fig.add_annotation(
        x=n_min_c,
        y=np.log10(err_min_c),
        text=f"МИНИМУМ<br>Chebyshev<br>n = {n_min_c}<br>err = {err_min_c:.2e}",
        showarrow=True,
        arrowhead=2,
        arrowsize=1.5,
        arrowwidth=2,
        arrowcolor="blue",
        ax=-40, ay=-60,
        bgcolor="rgba(255,255,255,0.5)",
        bordercolor="blue",
        borderwidth=2,
        borderpad=4,
        font=dict(size=13, color="darkblue", family="Arial Black"),
        align="left"
    )
    
    fig.update_layout(
        title=f"{prefix}: Максимальная ошибка vs n (с отмеченными минимумами)",
        xaxis_title="Количество узлов n",
        yaxis_title="Максимальная ошибка (log)",
        yaxis_type="log",
        template="ggplot2",
        hovermode="x unified",
        height=680,
        width=1150,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    )
    
    fig.update_yaxes(
        tickformat=".1e"
    )
    

    fig.show()


def plot_perturb_boxplot(filename, title):

    noise, err = load_perturb(filename)

    bins = [0.01, 0.02, 0.03, 0.04, 0.05]

    grouped = []

    for eps in bins:
        mask = (noise >= eps*0.5) & (noise <= eps*1.5)
        grouped.append(err[mask])

    x_vals = [np.mean(noise[(noise >= eps*0.5) & (noise <= eps*1.5)]) for eps in bins]

    fig = go.Figure()

    for i, eps in enumerate(bins):
        fig.add_trace(go.Box(
            x=[x_vals[i]]*len(grouped[i]),
            y=grouped[i],
            boxmean=True,
            name=f"eps = {100 * bins[i]:.0f}%<br>" +
                f"(x = {x_vals[i]:.5f})"
        ))

    fig.update_layout(
        title=title,
        xaxis_title="Фактическое возмущение данных",
        yaxis_title="Относительная ошибка",
        template="ggplot2",
        width=1000,
        height=600
    )
    
    fig.update_yaxes(
        tickformat=".0%"
    )
    
    fig.update_xaxes(
        tickformat=".2%"
    )
    
    fig.show()

In [217]:
nodes_list = [5, 7, 10]

dir = "../../data/"

In [218]:
plot_interpolation(dir, "f1", f1, nodes_list)

In [219]:
plot_errors_with_theory(dir, "f1", result_max.fun, nodes_list)

In [220]:
plot_error_vs_n(dir, "f1")

In [221]:
plot_interpolation(dir, "f2", f2, nodes_list)

In [222]:
plot_errors_with_theory(dir, "f2", 120, nodes_list)

In [223]:
plot_error_vs_n(dir, "f2")

In [224]:
plot_perturb_boxplot(dir + "f1_perturb_uniform.txt", "f1: uniform")

In [225]:
plot_perturb_boxplot(dir + "f1_perturb_cheb.txt", "f1: chebyshev")

In [226]:
plot_perturb_boxplot(dir + "f2_perturb_uniform.txt", "f2: uniform")

In [227]:
plot_perturb_boxplot(dir + "f2_perturb_cheb.txt", "f2: chebyshev")